In [3]:
import pandas as pd

# 1. Load your data
df = pd.read_csv('DFW_2026.csv', low_memory=False)

# 2. Count TOTAL homes sold by each broker
total_homes = df.groupby('list_broker_name_1').size().rename('Total_Listings')

# 3. Filter the data to ONLY look at new construction (2025 & 2026)
new_construction_df = df[df['yearBuilt'].isin([2025, 2026])]

# 4. Count NEW homes sold by each broker
new_homes = new_construction_df.groupby('list_broker_name_1').size().rename('New_Listings')

# 5. Combine these two counts into one table
builder_check = pd.concat([total_homes, new_homes], axis=1).fillna(0)

# 6. Calculate the percentage: (New Listings / Total Listings) * 100
builder_check['Percent_New'] = (builder_check['New_Listings'] / builder_check['Total_Listings']) * 100

# 7. Find the Homebuilders! 
# Let's say anyone with at least 5 listings, where 90%+ are new, is a builder.
homebuilders = builder_check[
    (builder_check['Total_Listings'] >= 5) & 
    (builder_check['Percent_New'] >= 90)
]

# Sort them from largest to smallest to see the biggest builders
homebuilders = homebuilders.sort_values('Total_Listings', ascending=False)

print(homebuilders)

                                Total_Listings  New_Listings  Percent_New
list_broker_name_1                                                       
Turner Mangum,LLC                         3268        2951.0    90.299878
Meritage Homes Realty                     1405        1356.0    96.512456
Jeanette Anderson Real Estate              980         887.0    90.510204
D.R. Horton, AMERICA'S Builder             823         802.0    97.448360
Escape Realty                              593         568.0    95.784148
HIGHLAND HOMES REALTY                      411         378.0    91.970803
Alexander Properties                       356         336.0    94.382022
Keller Williams Realty Lone St             175         172.0    98.285714
Master Key Realty                           32          32.0   100.000000
Bransom Real Estate                         21          20.0    95.238095


In [ ]:
# 1. Load your data with the low_memory fix to prevent the red warning
file_name = 'DFW_2026.csv'
df = pd.read_csv(file_name, low_memory=False)

# 2. Fix and calculate the "Days on Market" columns
df['listingAddedDate'] = pd.to_datetime(df['listingAddedDate'], errors='coerce')
df['lastSoldDate'] = pd.to_datetime(df['lastSoldDate'], errors='coerce')
df['days_on_market'] = (df['lastSoldDate'] - df['listingAddedDate']).dt.days

# 3. Separate the Builder logic
total_homes = df.groupby('list_broker_name_1').size().rename('Total_Listings')
new_construction_df = df[df['yearBuilt'].isin([2025, 2026])]
new_homes = new_construction_df.groupby('list_broker_name_1').size().rename('New_Listings')

# 4. Combine counts and calculate the "New Construction %"
builder_check = pd.concat([total_homes, new_homes], axis=1).fillna(0)
builder_check['Percent_New'] = (builder_check['New_Listings'] / builder_check['Total_Listings']) * 100

# 5. Define who a builder is (At least 5 listings, 90%+ are brand new)
builder_filter = (builder_check['Total_Listings'] >= 5) & (builder_check['Percent_New'] >= 90)
top_builders = builder_check[builder_filter].index

# 6. Pull average "Days on Market" ONLY for those specific builders
builders_only_df = df[df['list_broker_name_1'].isin(top_builders)]
avg_speeds = builders_only_df.groupby('list_broker_name_1')['days_on_market'].mean().rename('Avg_Days_On_Market')

# 7. Create the MASTER Dashboard
final_builder_dashboard = pd.concat([builder_check[builder_filter], avg_speeds], axis=1)

# Round the numbers neatly and sort by the fastest turnaround times
final_builder_dashboard['Percent_New'] = final_builder_dashboard['Percent_New'].round(1)
final_builder_dashboard['Avg_Days_On_Market'] = final_builder_dashboard['Avg_Days_On_Market'].round(1)
final_builder_dashboard = final_builder_dashboard.sort_values('Avg_Days_On_Market')

# Display the master table
final_builder_dashboard